# EEG Emotion Recognition - LOSO Pipeline

Овој notebook опфаќа:
- preprocessing на EEG сигнали
- екстракција на qEEG карактеристики
- SVM baseline
- EEGNet
- DeepConvNet
- LOSO евалуација

## 1. Импортирање на библиотеки

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io

from scipy.io import loadmat
from scipy.signal import butter, filtfilt, welch, resample

from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## 2. Helper functions

In [2]:
def bandpass_filter(data, lowcut=1, highcut=45, fs=256, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype='band')
    filtered = filtfilt(b, a, data, axis=-1)

    return filtered

In [3]:
def segment_signal(data, window_size=1024, step_size=512):
    """
    data shape: (channels, samples)
    returns shape: (num_segments, channels, window_size)
    """
    segments = []
    n_samples = data.shape[1]

    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)

    return np.array(segments)

In [4]:
def zscore_with_stats(signal, mean, std):
    return (signal - mean) / (std + 1e-8)

def compute_subject_stats(trials):
    """
    trials: list of arrays with shape [channels, time]
    returns mean/std per channel computed across all trials of one subject
    """
    all_data = np.concatenate(trials, axis=1)
    mean = np.mean(all_data, axis=1, keepdims=True)
    std = np.std(all_data, axis=1, keepdims=True)
    return mean, std

In [5]:
def resample_signal(signal, orig_fs, target_fs):
    num_samples = int(signal.shape[1] * target_fs / orig_fs)
    return resample(signal, num_samples, axis=1)

In [6]:
def extract_bandpower(segment, fs=256):
    bands = {
        'delta': (1, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 45)
    }

    features = []

    for ch in segment:
        freqs, psd = welch(ch, fs=fs)

        for band in bands.values():
            low, high = band
            idx = (freqs >= low) & (freqs <= high)
            band_power = np.mean(psd[idx])
            features.append(band_power)

    return np.array(features)

In [7]:
def extract_features_from_segments(segments, fs=256):
    feature_matrix = []

    for segment in segments:
        features = extract_bandpower(segment, fs=fs)
        feature_matrix.append(features)

    return np.array(feature_matrix)

In [8]:
def create_segment_labels(num_segments, trial_label):
    """
    num_segments: број на сегменти од едно recording/trial
    trial_label: 0 или 1
    returns shape: (num_segments,)
    """
    return np.full(num_segments, trial_label)

In [9]:
def create_subject_ids(num_segments, subject_id):
    """
    num_segments: број на сегменти
    subject_id: идентификатор на subject
    returns shape: (num_segments,)
    """
    return np.full(num_segments, subject_id)

In [10]:
def prepare_eegnet_data(segments):
    """
    segments shape: (num_segments, channels, samples)
    returns: (num_segments, 1, channels, samples)
    """
    return segments[:, np.newaxis, :, :]

In [11]:
def train_eegnet(model, X_train, y_train, epochs=10, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    return model

In [12]:
def evaluate_eegnet(model, X_test, y_test):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    with torch.no_grad():
        outputs = model(X_test)
        predictions = torch.argmax(outputs, dim=1).cpu().numpy()

    acc = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, zero_division=0)

    return acc, f1, predictions

In [13]:
def run_loso_svm(X, y, groups):
    logo = LeaveOneGroupOut()

    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    macro_f1_scores = []
    conf_matrices = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
        print(f"\nFold {fold}/{len(np.unique(groups))}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        print("Train class counts:", np.bincount(y_train))
        print("Test class counts:", np.bincount(y_test))

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = SVC(
            kernel='rbf',
            C=1.0,
            gamma='scale',
            class_weight='balanced'
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        cm = confusion_matrix(y_test, y_pred)

        print(f"  Fold Accuracy: {acc:.4f}")
        print(f"  Fold Precision: {prec:.4f}")
        print(f"  Fold Recall: {rec:.4f}")
        print(f"  Fold F1-score: {f1:.4f}")
        print(f"  Fold Macro F1: {f1_macro:.4f}")
        print(f"  Confusion Matrix:\n{cm}")

        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
        macro_f1_scores.append(f1_macro)
        conf_matrices.append(cm)

    return {
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std": np.std(accuracies),
        "precision_mean": np.mean(precisions),
        "precision_std": np.std(precisions),
        "recall_mean": np.mean(recalls),
        "recall_std": np.std(recalls),
        "f1_mean": np.mean(f1_scores),
        "f1_std": np.std(f1_scores),
        "macro_f1_mean": np.mean(macro_f1_scores),
        "macro_f1_std": np.std(macro_f1_scores),
        "confusion_matrices": conf_matrices
    }

In [14]:
def run_loso_eegnet(X, y, groups, num_channels=14, num_samples=1024, epochs=5, lr=0.001, batch_size=128):
    logo = LeaveOneGroupOut()

    accuracies = []
    f1_scores = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
        print(f"\nFold {fold}/{len(np.unique(groups))}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        X_test_t = torch.tensor(X_test, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.long)
        y_test_t = torch.tensor(y_test, dtype=torch.long)

        train_loader = DataLoader(
            TensorDataset(X_train_t, y_train_t),
            batch_size=batch_size,
            shuffle=True
        )

        test_loader = DataLoader(
            TensorDataset(X_test_t, y_test_t),
            batch_size=batch_size,
            shuffle=False
        )

        model = EEGNet(num_channels=num_channels, num_samples=num_samples, num_classes=2).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        model.train()
        for epoch in range(epochs):
            epoch_loss = 0.0

            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)

                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()

            print(f"  Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / len(train_loader):.4f}")

        model.eval()
        all_preds = []
        all_true = []

        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)

                outputs = model(batch_X)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                all_preds.extend(preds)
                all_true.extend(batch_y.numpy())

        acc = accuracy_score(all_true, all_preds)
        f1 = f1_score(all_true, all_preds, zero_division=0)

        accuracies.append(acc)
        f1_scores.append(f1)

        print(f"  Fold Accuracy: {acc:.4f}")
        print(f"  Fold F1-score: {f1:.4f}")

    return np.mean(accuracies), np.mean(f1_scores)

In [15]:
def resample_eeg(data, orig_fs=128, target_fs=256):
    """
    data shape: (channels, samples)
    """
    if orig_fs == target_fs:
        return data

    num_samples = int(data.shape[1] * target_fs / orig_fs)
    return resample(data, num_samples, axis=1)

## 3. Пример податоци за тестирање на pipeline

In [16]:
fake_eeg = np.random.randn(14, 5000)
segments = segment_signal(fake_eeg, window_size=1024, step_size=512)

print("Original shape:", fake_eeg.shape)
print("Segmented shape:", segments.shape)

Original shape: (14, 5000)
Segmented shape: (8, 14, 1024)


In [17]:
X_fake = extract_features_from_segments(segments, fs=256)
y_fake = create_segment_labels(num_segments=segments.shape[0], trial_label=1)
groups_fake = create_subject_ids(num_segments=segments.shape[0], subject_id=3)

print("Feature matrix shape:", X_fake.shape)
print("Label vector shape:", y_fake.shape)
print("Groups shape:", groups_fake.shape)

Feature matrix shape: (8, 70)
Label vector shape: (8,)
Groups shape: (8,)


## 4. SVM baseline

In [ ]:
# Fake data for 4 subjects
X_all = []
y_all = []
groups_all = []

for subject_id in range(4):
    fake_eeg = np.random.randn(14, 5000)
    segments = segment_signal(fake_eeg, window_size=1024, step_size=512)
    X_subject = extract_features_from_segments(segments, fs=256)

    num_segments = X_subject.shape[0]
    half = num_segments // 2
    y_subject = np.array([0] * half + [1] * (num_segments - half))
    g_subject = create_subject_ids(num_segments=num_segments, subject_id=subject_id)

    X_all.append(X_subject)
    y_all.append(y_subject)
    groups_all.append(g_subject)

X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)
groups_all = np.concatenate(groups_all)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)
print("groups_all shape:", groups_all.shape)
print("Unique labels:", np.unique(y_all))
print("Unique groups:", np.unique(groups_all))

X_all shape: (32, 70)
y_all shape: (32,)
groups_all shape: (32,)
Unique labels: [0 1]
Unique groups: [0 1 2 3]


## 5. EEGNet

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, num_channels=14, num_samples=1024, num_classes=2,
                 F1=8, D=2, F2=16, kernel_length=64, dropout=0.5):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),

            nn.Conv2d(F1, F1 * D, (num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout)
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_samples)
            out = self.block1(dummy)
            out = self.block2(out)
            flattened_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Linear(flattened_dim, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [18]:
X_eegnet = prepare_eegnet_data(segments)
print("EEGNet input shape:", X_eegnet.shape)

EEGNet input shape: (8, 1, 14, 1024)


In [ ]:
model = EEGNet()

x = torch.randn(8, 1, 14, 1024)
output = model(x)

print("Output shape:", output.shape)

Output shape: torch.Size([8, 2])


In [ ]:
y_eegnet = np.array([0, 0, 0, 0, 1, 1, 1, 1])

model = EEGNet(num_channels=14, num_samples=1024, num_classes=2)
trained_model = train_eegnet(model, X_eegnet, y_eegnet, epochs=5, lr=0.001)

Epoch 1/5, Loss: 0.7540
Epoch 2/5, Loss: 0.7109
Epoch 3/5, Loss: 0.8404
Epoch 4/5, Loss: 0.7044
Epoch 5/5, Loss: 0.7185


In [ ]:
acc, f1, preds = evaluate_eegnet(trained_model, X_eegnet, y_eegnet)

print("EEGNet Accuracy:", acc)
print("EEGNet F1-score:", f1)
print("Predictions:", preds)

EEGNet Accuracy: 0.75
EEGNet F1-score: 0.75
Predictions: [0 1 0 0 1 0 1 1]


In [ ]:
X_eeg_all = []
y_eeg_all = []
groups_eeg_all = []

for subject_id in range(4):
    fake_eeg = np.random.randn(14, 5000)
    segments = segment_signal(fake_eeg, window_size=1024, step_size=512)

    X_subject = prepare_eegnet_data(segments)

    num_segments = X_subject.shape[0]
    half = num_segments // 2
    y_subject = np.array([0] * half + [1] * (num_segments - half))
    g_subject = create_subject_ids(num_segments=num_segments, subject_id=subject_id)

    X_eeg_all.append(X_subject)
    y_eeg_all.append(y_subject)
    groups_eeg_all.append(g_subject)

X_eeg_all = np.vstack(X_eeg_all)
y_eeg_all = np.concatenate(y_eeg_all)
groups_eeg_all = np.concatenate(groups_eeg_all)

print("X_eeg_all shape:", X_eeg_all.shape)
print("y_eeg_all shape:", y_eeg_all.shape)
print("groups_eeg_all shape:", groups_eeg_all.shape)

X_eeg_all shape: (32, 1, 14, 1024)
y_eeg_all shape: (32,)
groups_eeg_all shape: (32,)


In [ ]:
mean_acc_eegnet, mean_f1_eegnet = run_loso_eegnet(
    X_eeg_all,
    y_eeg_all,
    groups_eeg_all,
    num_channels=14,
    num_samples=1024,
    epochs=5,
    lr=0.001
)

print("Mean LOSO EEGNet Accuracy:", mean_acc_eegnet)
print("Mean LOSO EEGNet F1-score:", mean_f1_eegnet)

Using device: cuda

Fold 1/4
  Epoch 1/5, Loss: 0.7310
  Epoch 2/5, Loss: 0.7064
  Epoch 3/5, Loss: 0.6538
  Epoch 4/5, Loss: 0.6566
  Epoch 5/5, Loss: 0.6777
  Fold Accuracy: 0.2500
  Fold F1-score: 0.4000

Fold 2/4
  Epoch 1/5, Loss: 0.7610
  Epoch 2/5, Loss: 0.7258
  Epoch 3/5, Loss: 0.7365
  Epoch 4/5, Loss: 0.6710
  Epoch 5/5, Loss: 0.6882
  Fold Accuracy: 0.5000
  Fold F1-score: 0.0000

Fold 3/4
  Epoch 1/5, Loss: 0.6921
  Epoch 2/5, Loss: 0.6937
  Epoch 3/5, Loss: 0.6445
  Epoch 4/5, Loss: 0.6516
  Epoch 5/5, Loss: 0.6431
  Fold Accuracy: 0.5000
  Fold F1-score: 0.6667

Fold 4/4
  Epoch 1/5, Loss: 0.7033
  Epoch 2/5, Loss: 0.7241
  Epoch 3/5, Loss: 0.6907
  Epoch 4/5, Loss: 0.6387
  Epoch 5/5, Loss: 0.7102
  Fold Accuracy: 1.0000
  Fold F1-score: 1.0000
Mean LOSO EEGNet Accuracy: 0.5625
Mean LOSO EEGNet F1-score: 0.5166666666666666


## 6. DeepConvNet

In [19]:
class DeepConvNet(nn.Module):
    def __init__(self, num_channels=14, num_samples=1024, num_classes=2):
        super(DeepConvNet, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 25, kernel_size=(1, 5)),
            nn.Conv2d(25, 25, kernel_size=(num_channels, 1)),
            nn.BatchNorm2d(25),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout(0.5)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(25, 50, kernel_size=(1, 5)),
            nn.BatchNorm2d(50),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout(0.5)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(50, 100, kernel_size=(1, 5)),
            nn.BatchNorm2d(100),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout(0.5)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(100, 200, kernel_size=(1, 5)),
            nn.BatchNorm2d(200),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout(0.5)
        )

        dummy = torch.randn(1, 1, num_channels, num_samples)
        dummy = self._forward_features(dummy)
        self.flattened_size = dummy.view(1, -1).shape[1]

        self.fc = nn.Linear(self.flattened_size, num_classes)

    def _forward_features(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [20]:
model = DeepConvNet()

x = torch.randn(8, 1, 14, 1024)
output = model(x)

print("DeepConvNet output shape:", output.shape)

DeepConvNet output shape: torch.Size([8, 2])


In [21]:
trained_model = train_eegnet(model, X_eegnet, y_eegnet, epochs=5, lr=0.001)

NameError: name 'y_eegnet' is not defined

In [22]:
acc, f1, preds = evaluate_eegnet(trained_model, X_eegnet, y_eegnet)

print("DeepConvNet Accuracy:", acc)
print("DeepConvNet F1:", f1)
print("Predictions:", preds)

NameError: name 'trained_model' is not defined

## 7. DREAMER dataset

### 8.1 Преземање на DREAMER

In [23]:
!pip install kagglehub

In [24]:
import kagglehub

path_dreamer = kagglehub.dataset_download("phhasian0710/dreamer")
print("DREAMER path:", path_dreamer)

100%|██████████| 847M/847M [00:12<00:00, 71.3MB/s]

Extracting files...


DREAMER path: /root/.cache/kagglehub/datasets/phhasian0710/dreamer/versions/1


### 8.2 Проверка на содржината

In [ ]:
import os

print(os.listdir(path_dreamer))

['dreamer', 'DREAMER.mat']


In [25]:
dreamer_file = os.path.join(path_dreamer, "DREAMER.mat")
print(dreamer_file)

/root/.cache/kagglehub/datasets/phhasian0710/dreamer/versions/1/DREAMER.mat


### 8.3 Вчитување на DREAMER.mat

In [26]:
import scipy.io

dreamer_data = scipy.io.loadmat(dreamer_file)
print(dreamer_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'DREAMER'])


In [27]:
import numpy as np

for key in dreamer_data.keys():
    if not key.startswith("__"):
        print(key, type(dreamer_data[key]), np.shape(dreamer_data[key]))

DREAMER <class 'numpy.ndarray'> (1, 1)


In [28]:
dreamer_main = dreamer_data["DREAMER"][0, 0]
print(type(dreamer_main))
print(dreamer_main.dtype)

<class 'numpy.void'>
[('Data', 'O'), ('EEG_SamplingRate', 'O'), ('ECG_SamplingRate', 'O'), ('EEG_Electrodes', 'O'), ('noOfSubjects', 'O'), ('noOfVideoSequences', 'O'), ('Disclaimer', 'O'), ('Provider', 'O'), ('Version', 'O'), ('Acknowledgement', 'O')]


In [29]:
print(dreamer_main.dtype.names)

('Data', 'EEG_SamplingRate', 'ECG_SamplingRate', 'EEG_Electrodes', 'noOfSubjects', 'noOfVideoSequences', 'Disclaimer', 'Provider', 'Version', 'Acknowledgement')


In [30]:
dreamer_subjects = dreamer_main["Data"]
print(type(dreamer_subjects))
print(np.shape(dreamer_subjects))

<class 'numpy.ndarray'>
(1, 23)


In [31]:
subject_1 = dreamer_subjects[0, 0]
print(type(subject_1))
print(subject_1.dtype)
print(subject_1.dtype.names)

<class 'numpy.ndarray'>
[('Age', 'O'), ('Gender', 'O'), ('EEG', 'O'), ('ECG', 'O'), ('ScoreValence', 'O'), ('ScoreArousal', 'O'), ('ScoreDominance', 'O')]
('Age', 'Gender', 'EEG', 'ECG', 'ScoreValence', 'ScoreArousal', 'ScoreDominance')


In [32]:
print("EEG sampling rate:", dreamer_main["EEG_SamplingRate"])
print("Number of subjects:", dreamer_main["noOfSubjects"])
print("Number of video sequences:", dreamer_main["noOfVideoSequences"])

EEG sampling rate: [[128]]
Number of subjects: [[23]]
Number of video sequences: [[18]]


In [33]:
subject_1 = dreamer_subjects[0, 0]

subject_1_eeg = subject_1["EEG"][0, 0]
subject_1_valence = subject_1["ScoreValence"][0, 0]

print("EEG type:", type(subject_1_eeg))
print("EEG shape:", np.shape(subject_1_eeg))

print("Valence type:", type(subject_1_valence))
print("Valence shape:", np.shape(subject_1_valence))
print("Valence scores:", subject_1_valence)

EEG type: <class 'numpy.ndarray'>
EEG shape: (1, 1)
Valence type: <class 'numpy.ndarray'>
Valence shape: (18, 1)
Valence scores: [[4]
 [3]
 [5]
 [4]
 [4]
 [1]
 [5]
 [1]
 [1]
 [5]
 [4]
 [4]
 [4]
 [3]
 [2]
 [3]
 [1]
 [3]]


In [34]:
trial_1_eeg = subject_1_eeg[0, 0]
trial_1_valence = subject_1_valence[0, 0]

print("Trial 1 EEG type:", type(trial_1_eeg))
print("Trial 1 EEG shape:", np.shape(trial_1_eeg))

print("Trial 1 valence:", trial_1_valence)

Trial 1 EEG type: <class 'numpy.void'>
Trial 1 EEG shape: ()
Trial 1 valence: 4


In [35]:
trial_1_eeg = subject_1["EEG"][0, 0]
print(type(trial_1_eeg))
print(trial_1_eeg.dtype)
print(trial_1_eeg.dtype.names)

<class 'numpy.ndarray'>
[('baseline', 'O'), ('stimuli', 'O')]
('baseline', 'stimuli')


In [36]:
for field in trial_1_eeg.dtype.names:
    value = trial_1_eeg[field]
    print(field, type(value), np.shape(value))

baseline <class 'numpy.ndarray'> (1, 1)
stimuli <class 'numpy.ndarray'> (1, 1)


In [37]:
subject_1_stimuli = subject_1["EEG"][0, 0]["stimuli"][0, 0]

print(type(subject_1_stimuli))
print(np.shape(subject_1_stimuli))

<class 'numpy.ndarray'>
(18, 1)


In [38]:
for i in range(3):
    trial = subject_1_stimuli[i, 0]
    print(f"Trial {i+1} type:", type(trial), "| shape:", np.shape(trial))

Trial 1 type: <class 'numpy.ndarray'> | shape: (25472, 14)
Trial 2 type: <class 'numpy.ndarray'> | shape: (16768, 14)
Trial 3 type: <class 'numpy.ndarray'> | shape: (44544, 14)


In [39]:
trial_1 = subject_1_stimuli[0, 0]
print("Trial 1 shape:", trial_1.shape)

print(trial_1[:3, :3])

Trial 1 shape: (25472, 14)
[[4388.20512821 4102.56410256 4219.48717949]
 [4375.8974359  4093.84615385 4252.82051282]
 [4378.46153846 4091.28205128 4230.25641026]]


### 8.4 Extraction of EEG trials and valence labels

In [40]:
def load_dreamer_subject(subject_data, label_threshold=3.0):
    """
    subject_data: one DREAMER subject object from dreamer_subjects[0, subject_id]
    returns:
        X_subject: (num_segments, channels, samples)
        y_subject: (num_segments,)
    """

    subject_eeg = subject_data["EEG"][0, 0]["stimuli"][0, 0]
    subject_scores = subject_data["ScoreValence"][0, 0].flatten()

    processed_trials = []
    trial_labels = []

    for i in range(subject_eeg.shape[0]):
        trial = subject_eeg[i, 0].T
        trial = bandpass_filter(trial, lowcut=1, highcut=45, fs=128, order=4)
        trial = resample_signal(trial, orig_fs=128, target_fs=256)

        processed_trials.append(trial)

        label = 1 if subject_scores[i] > label_threshold else 0
        trial_labels.append(label)

    mean, std = compute_subject_stats(processed_trials)

    X_subject = []
    y_subject = []

    for trial, label in zip(processed_trials, trial_labels):
        trial_norm = zscore_with_stats(trial, mean, std)
        segments = segment_signal(trial_norm, window_size=1024, step_size=512)

        X_subject.append(segments)
        y_subject.extend([label] * len(segments))

    X_subject = np.vstack(X_subject).astype(np.float32)
    y_subject = np.array(y_subject, dtype=np.int64)

    return X_subject, y_subject

In [41]:
subject_1 = dreamer_subjects[0, 0]

X_segments_sub1, y_labels_sub1 = load_dreamer_subject(subject_1, label_threshold=3)

print("Subject 1 segments shape:", X_segments_sub1.shape)
print("Subject 1 labels shape:", y_labels_sub1.shape)
print("Unique labels:", np.unique(y_labels_sub1))
print("Class counts:", np.bincount(y_labels_sub1))

Subject 1 segments shape: (1843, 14, 1024)
Subject 1 labels shape: (1843,)
Unique labels: [0 1]
Class counts: [977 866]


### 8.5 Подготовка на сите subjects за DREAMER

In [42]:
X_dreamer = []
y_dreamer = []
groups_dreamer = []

num_subjects = dreamer_subjects.shape[1]

for subject_id in range(num_subjects):
    subject = dreamer_subjects[0, subject_id]

    X_sub, y_sub = load_dreamer_subject(subject, label_threshold=3)
    g_sub = create_subject_ids(num_segments=len(y_sub), subject_id=subject_id)

    X_dreamer.append(X_sub)
    y_dreamer.append(y_sub)
    groups_dreamer.append(g_sub)

    print(f"Subject {subject_id+1}: segments={X_sub.shape[0]}, class counts={np.bincount(y_sub)}")

X_dreamer = np.vstack(X_dreamer)
y_dreamer = np.concatenate(y_dreamer)
groups_dreamer = np.concatenate(groups_dreamer)

print("\nFinal DREAMER shapes:")
print("X_dreamer:", X_dreamer.shape)
print("y_dreamer:", y_dreamer.shape)
print("groups_dreamer:", groups_dreamer.shape)
print("Unique labels:", np.unique(y_dreamer))
print("Unique groups:", np.unique(groups_dreamer))

Subject 1: segments=1843, class counts=[977 866]
Subject 2: segments=1843, class counts=[1256  587]
Subject 3: segments=1843, class counts=[943 900]
Subject 4: segments=1843, class counts=[1414  429]
Subject 5: segments=1843, class counts=[1158  685]
Subject 6: segments=1843, class counts=[1001  842]
Subject 7: segments=1843, class counts=[1207  636]
Subject 8: segments=1843, class counts=[1218  625]
Subject 9: segments=1843, class counts=[1029  814]
Subject 10: segments=1843, class counts=[ 695 1148]
Subject 11: segments=1843, class counts=[1189  654]
Subject 12: segments=1843, class counts=[901 942]
Subject 13: segments=1843, class counts=[ 804 1039]
Subject 14: segments=1843, class counts=[1024  819]
Subject 15: segments=1843, class counts=[1207  636]
Subject 16: segments=1843, class counts=[1160  683]
Subject 17: segments=1843, class counts=[1192  651]
Subject 18: segments=1843, class counts=[1160  683]
Subject 19: segments=1843, class counts=[876 967]
Subject 20: segments=1843, cl

### 8.6 LOSO евалуација на DREAMER со SVM

In [43]:
X_dreamer_features = extract_features_from_segments(X_dreamer, fs=256)
print("X_dreamer_features shape:", X_dreamer_features.shape)

KeyboardInterrupt: 

In [ ]:
np.save("/content/X_dreamer_features.npy", X_dreamer_features)
np.save("/content/y_dreamer.npy", y_dreamer)
np.save("/content/groups_dreamer.npy", groups_dreamer)

In [44]:
X_dreamer_features = np.load("X_dreamer_features.npy")
y_dreamer = np.load("y_dreamer.npy")
groups_dreamer = np.load("groups_dreamer.npy")

print("Loaded shapes:")
print(X_dreamer_features.shape)
print(y_dreamer.shape)
print(groups_dreamer.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'X_dreamer_features.npy'

In [46]:
svm_results = run_loso_svm(
    X_dreamer_features,
    y_dreamer,
    groups_dreamer
)

print("\n===== DREAMER LOSO SVM RESULTS =====")
print(f"Accuracy:  {svm_results['accuracy_mean']:.4f} ± {svm_results['accuracy_std']:.4f}")
print(f"Precision: {svm_results['precision_mean']:.4f} ± {svm_results['precision_std']:.4f}")
print(f"Recall:    {svm_results['recall_mean']:.4f} ± {svm_results['recall_std']:.4f}")
print(f"F1-score:  {svm_results['f1_mean']:.4f} ± {svm_results['f1_std']:.4f}")
print(f"Macro F1:  {svm_results['macro_f1_mean']:.4f} ± {svm_results['macro_f1_std']:.4f}")

NameError: name 'X_dreamer_features' is not defined

In [ ]:
results = {
    "accuracy_mean": svm_results["accuracy_mean"],
    "accuracy_std": svm_results["accuracy_std"],
    "precision_mean": svm_results["precision_mean"],
    "precision_std": svm_results["precision_std"],
    "recall_mean": svm_results["recall_mean"],
    "recall_std": svm_results["recall_std"],
    "f1_mean": svm_results["f1_mean"],
    "f1_std": svm_results["f1_std"],
    "macro_f1_mean": svm_results["macro_f1_mean"],
    "macro_f1_std": svm_results["macro_f1_std"],
}

np.save("dreamer_svm_results.npy", results)

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [48]:
import os

save_dir = "/content/drive/MyDrive/diplomska_eeg"
os.makedirs(save_dir, exist_ok=True)
print(save_dir)

/content/drive/MyDrive/diplomska_eeg


In [49]:
np.save(os.path.join(save_dir, "X_dreamer_features.npy"), X_dreamer_features)
np.save(os.path.join(save_dir, "y_dreamer.npy"), y_dreamer)
np.save(os.path.join(save_dir, "groups_dreamer.npy"), groups_dreamer)

NameError: name 'X_dreamer_features' is not defined

## 9. EEGNet LOSO on DREAMER

In [50]:
import os

save_dir_raw = "/content/drive/MyDrive/diplomska_eeg_raw"
os.makedirs(save_dir_raw, exist_ok=True)

print(save_dir_raw)

/content/drive/MyDrive/diplomska_eeg_raw


In [51]:
num_subjects = dreamer_subjects.shape[1]

for subject_id in range(num_subjects):
    subject = dreamer_subjects[0, subject_id]

    X_sub, y_sub = load_dreamer_subject(subject, label_threshold=3)

    X_sub = X_sub.astype(np.float32)
    y_sub = y_sub.astype(np.int8)

    np.save(os.path.join(save_dir_raw, f"X_subject_{subject_id:02d}.npy"), X_sub)
    np.save(os.path.join(save_dir_raw, f"y_subject_{subject_id:02d}.npy"), y_sub)

    print(f"Saved subject {subject_id+1}/{num_subjects} -> X: {X_sub.shape}, y: {y_sub.shape}, class counts: {np.bincount(y_sub)}")

    del X_sub, y_sub
    gc.collect()

Saved subject 1/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [977 866]
Saved subject 2/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [1256  587]
Saved subject 3/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [943 900]
Saved subject 4/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [1414  429]
Saved subject 5/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [1158  685]
Saved subject 6/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [1001  842]
Saved subject 7/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [1207  636]
Saved subject 8/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [1218  625]
Saved subject 9/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [1029  814]
Saved subject 10/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [ 695 1148]
Saved subject 11/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [1189  654]
Saved subject 12/23 -> X: (1843, 14, 1024), y: (1843,), class counts: [901 942]
Saved subject 13/23 -> X: (1843

In [ ]:
def run_loso_eegnet_from_disk(save_dir, num_subjects=23, num_channels=14, num_samples=1024,
                              epochs=3, lr=0.001, batch_size=32):
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    macro_f1_scores = []
    conf_matrices = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    for test_subject in range(num_subjects):
        print(f"\nFold {test_subject+1}/{num_subjects}")

        train_subjects = [i for i in range(num_subjects) if i != test_subject]

        # test data
        X_test = np.load(os.path.join(save_dir, f"X_subject_{test_subject:02d}.npy"))
        y_test = np.load(os.path.join(save_dir, f"y_subject_{test_subject:02d}.npy"))

        print("Test class counts:", np.bincount(y_test))

        # class weights from train labels only
        y_train_all = []
        for subject_id in train_subjects:
            y_sub = np.load(os.path.join(save_dir, f"y_subject_{subject_id:02d}.npy"))
            y_train_all.append(y_sub)

        y_train_all = np.concatenate(y_train_all)
        print("Train class counts:", np.bincount(y_train_all))

        classes = np.array([0, 1])
        weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_all)
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        model = EEGNet(
            num_channels=num_channels,
            num_samples=num_samples,
            num_classes=2,
            F1=8,
            D=2,
            F2=16,
            kernel_length=64,
            dropout=0.5
        ).to(device)

        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            batch_count = 0

            for subject_id in train_subjects:
                X_sub = np.load(os.path.join(save_dir, f"X_subject_{subject_id:02d}.npy"), mmap_mode='r')
                y_sub = np.load(os.path.join(save_dir, f"y_subject_{subject_id:02d}.npy"), mmap_mode='r')

                X_sub = np.array(X_sub, dtype=np.float32, copy=True)
                y_sub = np.array(y_sub, dtype=np.int64, copy=True)

                X_sub = X_sub[:, np.newaxis, :, :]
                train_ds = TensorDataset(torch.from_numpy(X_sub), torch.from_numpy(y_sub))
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

                for xb, yb in train_loader:
                    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

                    optimizer.zero_grad()
                    outputs = model(xb)
                    loss = criterion(outputs, yb)
                    loss.backward()
                    optimizer.step()

                    epoch_loss += loss.item()
                    batch_count += 1

                del X_sub, y_sub, train_ds, train_loader
                gc.collect()

            print(f"  Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / max(batch_count, 1):.4f}")

        model.eval()
        X_test = X_test[:, np.newaxis, :, :].astype(np.float32)
        test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test.astype(np.int64)))
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

        all_preds = []
        all_true = []

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device, non_blocking=True)
                outputs = model(xb)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                all_preds.extend(preds)
                all_true.extend(yb.numpy())

        all_preds = np.array(all_preds)
        all_true = np.array(all_true)

        acc = accuracy_score(all_true, all_preds)
        prec = precision_score(all_true, all_preds, zero_division=0)
        rec = recall_score(all_true, all_preds, zero_division=0)
        f1 = f1_score(all_true, all_preds, zero_division=0)
        f1_macro = f1_score(all_true, all_preds, average='macro', zero_division=0)
        cm = confusion_matrix(all_true, all_preds)

        print(f"  Fold Accuracy: {acc:.4f}")
        print(f"  Fold Precision: {prec:.4f}")
        print(f"  Fold Recall: {rec:.4f}")
        print(f"  Fold F1-score: {f1:.4f}")
        print(f"  Fold Macro F1: {f1_macro:.4f}")
        print(f"  Confusion Matrix:\n{cm}")

        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
        macro_f1_scores.append(f1_macro)
        conf_matrices.append(cm)

        del X_test, y_test, test_ds, test_loader, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std": np.std(accuracies),
        "precision_mean": np.mean(precisions),
        "precision_std": np.std(precisions),
        "recall_mean": np.mean(recalls),
        "recall_std": np.std(recalls),
        "f1_mean": np.mean(f1_scores),
        "f1_std": np.std(f1_scores),
        "macro_f1_mean": np.mean(macro_f1_scores),
        "macro_f1_std": np.std(macro_f1_scores),
        "confusion_matrices": conf_matrices
    }

In [ ]:
eegnet_results = run_loso_eegnet_from_disk(
    save_dir=save_dir_raw,
    num_subjects=23,
    num_channels=14,
    num_samples=1024,
    epochs=5,
    batch_size=64,
    lr=0.001,
)

print("\n===== DREAMER LOSO EEGNet RESULTS =====")
print(f"Accuracy:  {eegnet_results['accuracy_mean']:.4f} ± {eegnet_results['accuracy_std']:.4f}")
print(f"Precision: {eegnet_results['precision_mean']:.4f} ± {eegnet_results['precision_std']:.4f}")
print(f"Recall:    {eegnet_results['recall_mean']:.4f} ± {eegnet_results['recall_std']:.4f}")
print(f"F1-score:  {eegnet_results['f1_mean']:.4f} ± {eegnet_results['f1_std']:.4f}")
print(f"Macro F1:  {eegnet_results['macro_f1_mean']:.4f} ± {eegnet_results['macro_f1_std']:.4f}")

Using device: cuda

Fold 1/23
Test class counts: [977 866]
Train class counts: [23984 16562]
  Epoch 1/5, Loss: 0.6924
  Epoch 2/5, Loss: 0.6758
  Epoch 3/5, Loss: 0.6711
  Epoch 4/5, Loss: 0.6605
  Epoch 5/5, Loss: 0.6580
  Fold Accuracy: 0.4835
  Fold Precision: 0.0700
  Fold Recall: 0.0081
  Fold F1-score: 0.0145
  Fold Macro F1: 0.3322
  Confusion Matrix:
[[884  93]
 [859   7]]

Fold 2/23
Test class counts: [1256  587]
Train class counts: [23705 16841]
  Epoch 1/5, Loss: 0.6945
  Epoch 2/5, Loss: 0.6860
  Epoch 3/5, Loss: 0.6670
  Epoch 4/5, Loss: 0.6593
  Epoch 5/5, Loss: 0.6439
  Fold Accuracy: 0.3277
  Fold Precision: 0.3078
  Fold Recall: 0.8893
  Fold F1-score: 0.4573
  Fold Macro F1: 0.2871
  Confusion Matrix:
[[  82 1174]
 [  65  522]]

Fold 3/23
Test class counts: [943 900]
Train class counts: [24018 16528]
  Epoch 1/5, Loss: 0.6828
  Epoch 2/5, Loss: 0.6628
  Epoch 3/5, Loss: 0.6536
  Epoch 4/5, Loss: 0.6467
  Epoch 5/5, Loss: 0.6432
  Fold Accuracy: 0.5339
  Fold Precisio

In [ ]:
np.save("dreamer_eegnet_results.npy", eegnet_results)
print("EEGNet results saved.")

EEGNet results saved.


## 10. DeepConvNet LOSO on DREAMER

In [55]:
import shutil

src = "/content/drive/MyDrive/diplomska_eeg_raw"
dst = "/content/diplomska_eeg_raw"

shutil.copytree(src, dst, dirs_exist_ok=True)

print("Copied to local /content")

Copied to local /content


In [57]:
save_dir_raw = "/content/diplomska_eeg_raw"
print(save_dir_raw)

/content/diplomska_eeg_raw


In [58]:
class DeepConvNet(nn.Module):
    def __init__(self, num_channels=14, num_samples=1024, num_classes=2, dropout=0.5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 25, (1, 10), stride=1, padding=0, bias=False),
            nn.Conv2d(25, 25, (num_channels, 1), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(25),
            nn.ELU(),
            nn.MaxPool2d((1, 3)),
            nn.Dropout(dropout),

            nn.Conv2d(25, 50, (1, 10), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(50),
            nn.ELU(),
            nn.MaxPool2d((1, 3)),
            nn.Dropout(dropout),

            nn.Conv2d(50, 100, (1, 10), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(100),
            nn.ELU(),
            nn.MaxPool2d((1, 3)),
            nn.Dropout(dropout),

            nn.Conv2d(100, 200, (1, 10), stride=1, padding=0, bias=False),
            nn.BatchNorm2d(200),
            nn.ELU(),
            nn.MaxPool2d((1, 3)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_samples)
            out = self.features(dummy)
            flattened_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Linear(flattened_dim, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [59]:
def run_loso_deepconvnet_from_disk(save_dir, num_subjects=23, num_channels=14, num_samples=1024,
                                   epochs=3, lr=0.001, batch_size=64):
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    macro_f1_scores = []
    conf_matrices = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    for test_subject in range(num_subjects):
        print(f"\nFold {test_subject+1}/{num_subjects}")

        train_subjects = [i for i in range(num_subjects) if i != test_subject]

        X_test = np.load(os.path.join(save_dir, f"X_subject_{test_subject:02d}.npy"))
        y_test = np.load(os.path.join(save_dir, f"y_subject_{test_subject:02d}.npy"))

        print("Test class counts:", np.bincount(y_test))

        y_train_all = []
        for subject_id in train_subjects:
            y_sub = np.load(os.path.join(save_dir, f"y_subject_{subject_id:02d}.npy"))
            y_train_all.append(y_sub)

        y_train_all = np.concatenate(y_train_all)
        print("Train class counts:", np.bincount(y_train_all))

        classes = np.array([0, 1])
        weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_all)
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        model = DeepConvNet(
            num_channels=num_channels,
            num_samples=num_samples,
            num_classes=2,
            dropout=0.5
        ).to(device)

        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            batch_count = 0

            for subject_id in train_subjects:
                X_sub = np.load(os.path.join(save_dir, f"X_subject_{subject_id:02d}.npy"), mmap_mode='r')
                y_sub = np.load(os.path.join(save_dir, f"y_subject_{subject_id:02d}.npy"), mmap_mode='r')

                X_sub = np.array(X_sub, dtype=np.float32, copy=True)
                y_sub = np.array(y_sub, dtype=np.int64, copy=True)

                X_sub = X_sub[:, np.newaxis, :, :]
                train_ds = TensorDataset(torch.from_numpy(X_sub), torch.from_numpy(y_sub))
                train_loader = DataLoader(
                    train_ds,
                    batch_size=batch_size,
                    shuffle=True,
                    num_workers=0,
                    pin_memory=True
                )

                for xb, yb in train_loader:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)

                    optimizer.zero_grad()
                    outputs = model(xb)
                    loss = criterion(outputs, yb)
                    loss.backward()
                    optimizer.step()

                    epoch_loss += loss.item()
                    batch_count += 1

                del X_sub, y_sub, train_ds, train_loader
                gc.collect()

            print(f"  Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / max(batch_count, 1):.4f}")

        model.eval()
        X_test = X_test[:, np.newaxis, :, :].astype(np.float32)
        test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test.astype(np.int64)))
        test_loader = DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )

        all_preds = []
        all_true = []

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device, non_blocking=True)
                outputs = model(xb)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                all_preds.extend(preds)
                all_true.extend(yb.numpy())

        all_preds = np.array(all_preds)
        all_true = np.array(all_true)

        acc = accuracy_score(all_true, all_preds)
        prec = precision_score(all_true, all_preds, zero_division=0)
        rec = recall_score(all_true, all_preds, zero_division=0)
        f1 = f1_score(all_true, all_preds, zero_division=0)
        f1_macro = f1_score(all_true, all_preds, average='macro', zero_division=0)
        cm = confusion_matrix(all_true, all_preds)

        print(f"  Fold Accuracy: {acc:.4f}")
        print(f"  Fold Precision: {prec:.4f}")
        print(f"  Fold Recall: {rec:.4f}")
        print(f"  Fold F1-score: {f1:.4f}")
        print(f"  Fold Macro F1: {f1_macro:.4f}")
        print(f"  Confusion Matrix:\n{cm}")

        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
        macro_f1_scores.append(f1_macro)
        conf_matrices.append(cm)

        del X_test, y_test, test_ds, test_loader, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std": np.std(accuracies),
        "precision_mean": np.mean(precisions),
        "precision_std": np.std(precisions),
        "recall_mean": np.mean(recalls),
        "recall_std": np.std(recalls),
        "f1_mean": np.mean(f1_scores),
        "f1_std": np.std(f1_scores),
        "macro_f1_mean": np.mean(macro_f1_scores),
        "macro_f1_std": np.std(macro_f1_scores),
        "confusion_matrices": conf_matrices
    }

In [60]:
deepconv_results = run_loso_deepconvnet_from_disk(
    save_dir=save_dir_raw,
    num_subjects=23,
    num_channels=14,
    num_samples=1024,
    epochs=3,
    batch_size=64,
    lr=0.001,
)

print("\n===== DREAMER LOSO DeepConvNet RESULTS =====")
print(f"Accuracy:  {deepconv_results['accuracy_mean']:.4f} ± {deepconv_results['accuracy_std']:.4f}")
print(f"Precision: {deepconv_results['precision_mean']:.4f} ± {deepconv_results['precision_std']:.4f}")
print(f"Recall:    {deepconv_results['recall_mean']:.4f} ± {deepconv_results['recall_std']:.4f}")
print(f"F1-score:  {deepconv_results['f1_mean']:.4f} ± {deepconv_results['f1_std']:.4f}")
print(f"Macro F1:  {deepconv_results['macro_f1_mean']:.4f} ± {deepconv_results['macro_f1_std']:.4f}")

Using device: cuda

Fold 1/23
Test class counts: [977 866]
Train class counts: [23984 16562]
  Epoch 1/3, Loss: 0.7181
  Epoch 2/3, Loss: 0.6956
  Epoch 3/3, Loss: 0.6734
  Fold Accuracy: 0.5258
  Fold Precision: 0.3750
  Fold Recall: 0.0139
  Fold F1-score: 0.0267
  Fold Macro F1: 0.3566
  Confusion Matrix:
[[957  20]
 [854  12]]

Fold 2/23
Test class counts: [1256  587]
Train class counts: [23705 16841]
  Epoch 1/3, Loss: 0.7034
  Epoch 2/3, Loss: 0.6974
  Epoch 3/3, Loss: 0.6841
  Fold Accuracy: 0.6034
  Fold Precision: 0.3085
  Fold Recall: 0.1976
  Fold F1-score: 0.2409
  Fold Macro F1: 0.4862
  Confusion Matrix:
[[996 260]
 [471 116]]

Fold 3/23
Test class counts: [943 900]
Train class counts: [24018 16528]
  Epoch 1/3, Loss: 0.7051
  Epoch 2/3, Loss: 0.7080
  Epoch 3/3, Loss: 0.6965
  Fold Accuracy: 0.6164
  Fold Precision: 0.7149
  Fold Recall: 0.3567
  Fold F1-score: 0.4759
  Fold Macro F1: 0.5867
  Confusion Matrix:
[[815 128]
 [579 321]]

Fold 4/23
Test class counts: [1414  